# Setup

In [1]:
# Import libs
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import torch

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

from tqdm.auto import tqdm

from scipy.stats import wilcoxon
from lifelines import CoxPHFitter

from IPython.display import display, Markdown


In [2]:
# Set seed
sc.settings.verbosity = 3
sc.settings.seed = 0
np.random.seed(0)

In [3]:
# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())

# Check the name of the GPU
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA A100-SXM4-80GB


In [4]:
# Colours
import colorcet as cc
colors3 = cc.palette["glasbey"][:3]



In [5]:
# Save plot dir
output_dir = '/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Plots/'
sc.settings.figdir = output_dir

In [6]:
# Set scanpy plotting defaults
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

# TLS crop generation

## Data loading

In [ ]:
combined_dir = "/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Subclustering_analysis/combined_for_fig1"
crop_dir = "/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Subclustering_analysis/adata_TLS_cropped/all"
os.makedirs(crop_dir, exist_ok=True)

In [ ]:
adata_whole = sc.read_h5ad(f"{combined_dir}/adata_whole_combined.h5ad")
print(f"Loaded adata_whole: {adata_whole.shape[0]} cells, {adata_whole.shape[1]} genes")

adata_whole.obs["batch_filename"] = adata_whole.obs["batch"].astype(str).apply(os.path.basename)
print("Unique batch_filename values in adata_whole:")
print(sorted(adata_whole.obs["batch_filename"].unique()))

## Generate and save crops

In [ ]:
crops = {
    "CV6-KID-0-FT-1.h5ad":    (2500, 4000,  500, 2000),
    "CV9-KID-0-FT-2.h5ad":    (2400, 3500, 3500, 5200),
    "DI10-KID-0-FT-3.h5ad":   (2500, 3500, 1500, 2600),
    "DI13-KID-0-FT-1.h5ad":   ( 600, 1100, 2300, 2900),
    "CV1-KID-0-FO-1.h5ad":    ( 300, 4100, 2000, 3800),
    "CV1-KID-0-FT-2.h5ad":    (1500, 3500, 1000, 2500),
    "CV5-KID-0-FO-1.h5ad":    (3900, 5500,  700, 3400),
    "CV7-KID-0-FT-2-s3.h5ad": ( 100, 1200,  200, 1700),
}

In [ ]:
for fname, (x1, x2, y1, y2) in crops.items():
    section_mask = (adata_whole.obs["batch_filename"] == fname).to_numpy()
    if section_mask.sum() == 0:
        print(f"[skip] {fname}: no cells in adata_whole")
        continue

    adata_section = adata_whole[section_mask].copy()

    x = adata_section.obs["x_centroid"].to_numpy()
    y = adata_section.obs["y_centroid"].to_numpy()
    crop_mask = (x > x1) & (x < x2) & (y > y1) & (y < y2)
    adata_crop = adata_section[crop_mask].copy()
    print(f"{fname}: cropped {adata_crop.n_obs} / {adata_section.n_obs} cells")

    adata_crop.uns = {}
    sq.gr.spatial_neighbors(
        adata=adata_crop,
        spatial_key="spatial",
        library_key=None,
        set_diag=False,
        delaunay=False,
        n_neighs=5,
    )

    out_name = fname.replace(".h5ad", "_crop.h5ad")
    out_path = os.path.join(crop_dir, out_name)
    adata_crop.write_h5ad(out_path)
    print(f"  saved -> {out_path}")

# T cell replacement

## Perturbation analysis

In [19]:

ROOT = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Perturbations/Control_workflow')
CACHE = ROOT / 'mean_generated_adatas'
GEN_KEY = 'MintFLow_Generated_Xmic'
L4 = 'level_4_cell_type'
 
EXP_ORDER = [
    'shared_unperturbed', 'dose_response_25pct', 'dose_response_50pct',
    'dose_response_75pct', 'full_replacement', 'random_labels_weighted',
]
SECTIONS = [
    'CV1-KID-0-FT-2', 'CV6-KID-0-FT-1', 'CV7-KID-0-FT-2-s3', 'CV9-KID-0-FT-2',
    'CV1-KID-0-FO-1', 'DI10-KID-0-FT-3', 'DI13-KID-0-FT-1', 'CV5-KID-0-FO-1',
]
 

In [ ]:
### The perturbation experiment is repeated 10 times with different random seeds
def get_run_dirs(root=ROOT):
    return sorted(p for p in root.iterdir() if p.is_dir() and p.name.startswith('run_'))


def collect_generated_h5ad_map(run_dir, experiment_name):
    return {p.name: p for p in sorted((Path(run_dir) / experiment_name / 'generated_adatas').glob('*.h5ad'))}


def build_mean_generated_adata(source_paths, obsm_key=GEN_KEY):
    source_paths = [Path(p) for p in source_paths]
    template = sc.read_h5ad(source_paths[0]).copy()
    expr_sum = np.asarray(template.obsm[obsm_key], dtype=np.float64).copy()
    for p in source_paths[1:]:
        expr_sum += np.asarray(sc.read_h5ad(p).obsm[obsm_key], dtype=np.float64)
    template.obsm[obsm_key] = (expr_sum / len(source_paths)).astype(
        np.asarray(template.obsm[obsm_key]).dtype, copy=False
    )
    return template


def save_aggregated(mean_generated_adatas, cache_dir=CACHE):
    for exp_name, file_dict in mean_generated_adatas.items():
        exp_dir = cache_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)
        for filename, adata in file_dict.items():
            adata.write_h5ad(exp_dir / filename)
    print(f'Saved aggregated adatas to {cache_dir}')


def load_aggregated(cache_dir=CACHE):
    result = {}
    for exp_dir in sorted(cache_dir.iterdir()):
        if exp_dir.is_dir():
            result[exp_dir.name] = {p.name: sc.read_h5ad(p) for p in sorted(exp_dir.glob('*.h5ad'))}
    print(f'Loaded aggregated adatas from {cache_dir} ({len(result)} experiments)')
    return result


def list_experiment_names(run_dirs, preferred_order=None):
    preferred_order = preferred_order or []
    experiment_names = sorted(
        set.intersection(
            *[
                {
                    path.name
                    for path in run_dir.iterdir()
                    if path.is_dir() and (path / 'generated_adatas').exists()
                }
                for run_dir in run_dirs
            ]
        )
    )
    ordered_names = [name for name in preferred_order if name in experiment_names]
    ordered_names.extend(name for name in experiment_names if name not in ordered_names)
    return ordered_names


# --- Load from cache, or build and save if cache is missing ---

if CACHE.exists() and any(CACHE.iterdir()):
    mean_adatas = load_aggregated()
else:
    run_dirs = get_run_dirs()
    run_names = [run_dir.name for run_dir in run_dirs]
    experiment_names = list_experiment_names(run_dirs, preferred_order=EXP_ORDER)

    mean_adatas = {}
    for experiment_name in experiment_names:
        run_file_maps = {
            run_name: collect_generated_h5ad_map(run_dir, experiment_name)
            for run_name, run_dir in zip(run_names, run_dirs)
        }
        all_filenames = sorted(set.intersection(*(set(file_map) for file_map in run_file_maps.values())))
        mean_adatas[experiment_name] = {}
        for filename in tqdm(all_filenames, desc=experiment_name, leave=False):
            mean_adata = build_mean_generated_adata(
                [run_file_maps[run_name][filename] for run_name in run_names]
            )
            mean_adatas[experiment_name][filename] = mean_adata

    save_aggregated(mean_adatas)


In [21]:

mac = sc.read_h5ad('/nfs/team361/cl35/Macs/adata_macrophages.h5ad')
 
untreated = {
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV1-KID-0-FT-2.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV9-KID-0-FT-2.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV6-KID-0-FT-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/unannotated_data/CV7-KID-0-FT-2-s3.h5ad',
}
treated = {
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/DI13-KID-0-FT-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/DI10-KID-0-FT-3.h5ad',
}
 
obs = mac[mac.obs['batch'].astype(str).isin(untreated | treated)].copy()
obs.obs[L4] = obs.obs['Annotation_TAMs_merged'].astype(str).replace({'nan': 'NA', 'None': 'NA'}).values
obs.obs['treatment'] = np.where(obs.obs['batch'].astype(str).isin(treated), 'treated', 'untreated')
 
upreg_genes = {}
for label in sorted(obs.obs[L4].unique()):
    sub = obs[obs.obs[L4].eq(label)].copy()
    if min(sub.obs['treatment'].eq('treated').sum(), sub.obs['treatment'].eq('untreated').sum()) < 2:
        continue
    sub.X = (sub.layers['counts'] if 'counts' in sub.layers else sub.X).copy()
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    sc.tl.rank_genes_groups(sub, 'treatment', groups=['treated'], reference='untreated',
                            method='wilcoxon', pts=True, n_genes=sub.n_vars)
    df = sc.get.rank_genes_groups_df(sub, group='treated')
    genes = df.loc[(df['logfoldchanges'] > 0) & (df['pvals_adj'] < 0.05), 'names'].tolist()
    if genes:
        upreg_genes[label] = genes
 
del mac, obs
print(f'Upreg DEG signatures for {len(upreg_genes)} cell types')
for label, genes in upreg_genes.items():
    print(f'  {label} ({len(genes)} genes): {genes}')
 

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


normalizing counts per cell


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


    finished (0:00:00)
ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
normalizing counts per cell
    finished (0:00:00)
ranking genes


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)
normalizing counts per cell
    finished (0:00:00)
ranking genes


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)
normalizing counts per cell
    finished (0:00:00)
ranking genes


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
Upreg DEG signatures for 8 cell types
  FOLR2+ (100 genes): ['CD163', 'HOXB7', 'CD14', 'F13A1', 'CYP1B1', 'STAB1', 'SEPTIN9', 'SOST', 'RNF187', 'CD4', 'SPON2', 'PLTP', 'P4HB', 'CYBA', 'ANXA2', 'ALDOA', 'VEGFB', 'TMEM173', 'PGK1', 'MVP', 'FSTL1', 'PGF', 'ZNF683', 'REN', 'AOC3', 'STC2', 'LAMB3', 'BNIP3', 'C4A', 'SNAI2', 'FGFR4', 'FABP7', 'MRC1', 'COL4A1', 'FRZB', 'SERPINA3', 'TNXB', 'SLC2A1', 'SULF2', 'PRKAR1A', 'TMEM45A', 'C4B', 'CALM3', 'FCGBP', 'TNFSF12', 'CP', 'PYCARD', 'FES', 'ST14', 'SOD1', 'NREP', 'WAS', 'LBP', 'COL4A2', 'ENPP3', 'CLEC4M', 'NBL1', 'TSPAN9', 'TGFB1', 'COL5A1', 'IGFBP1', 'LAD1', 'LGALS9', 'ITGB5', 'ABCC

In [22]:

pseudo = 1e-9
 
def mean_sig_by_celltype(adata, agg='mean'):
    """Aggregate raw expression of signature genes per cell type."""
    a = adata.copy()
    a.X = np.asarray(a.obsm[GEN_KEY], dtype=np.float64)
    a.obs[L4] = a.obs[L4].astype(str).replace({'nan': 'NA', 'None': 'NA'}).values
    results = {}
    for label, genes in upreg_genes.items():
        mask = a.obs[L4].eq(label)
        present = [g for g in genes if g in a.var_names]
        if not mask.any() or not present:
            continue
        sub = a[mask, present]
        vals = np.clip(sub.X, 0, None)
        results[label] = float(np.median(vals) if agg == 'median' else np.mean(vals))
    return results
 
# Build raw mean expression table from mean adatas
raw_rows = []
for exp in EXP_ORDER:
    if exp not in mean_adatas:
        continue
    for fname, adata in mean_adatas[exp].items():
        section = Path(fname).stem.rsplit('__', 1)[0]
        if section not in SECTIONS:
            continue
        for label, val in mean_sig_by_celltype(adata).items():
            raw_rows.append({'experiment': exp, 'section': section, L4: label, 'mean_raw': val})
 
raw_df = pd.DataFrame(raw_rows)
 
# Compute log2FC relative to unperturbed per section
unpert = (
    raw_df[raw_df['experiment'] == 'shared_unperturbed']
    .set_index(['section', L4])['mean_raw']
)
 
lfc_rows = []
for _, row in raw_df.iterrows():
    key = (row['section'], row[L4])
    if key not in unpert.index:
        continue
    base = unpert[key]
    if row['experiment'] == 'shared_unperturbed':
        lfc_rows.append({'experiment': row['experiment'], 'section': row['section'],
                         L4: row[L4], 'log2FC': 0.0})
    else:
        lfc = np.log2((row['mean_raw'] + pseudo) / (base + pseudo))
        lfc_rows.append({'experiment': row['experiment'], 'section': row['section'],
                         L4: row[L4], 'log2FC': lfc})
 
lfc_df = pd.DataFrame(lfc_rows)
 
# Average across sections
plot_data = (
    lfc_df.groupby(['experiment', L4])['log2FC']
    .agg(['mean', 'sem']).reset_index()
)
plot_data.columns = ['experiment', L4, 'mean_lfc', 'sem_lfc']
plot_data['sem_lfc'] = plot_data['sem_lfc'].fillna(0.0)
 

In [23]:

wanted = ['IFN+', 'SPP1+GRN+', 'FOLR2+', 'IL1B+FCGR2A+']
pdf = plot_data[plot_data[L4].isin(wanted)]
 
present_exps = [e for e in EXP_ORDER if e in pdf['experiment'].unique()]
exp_pos = {e: i for i, e in enumerate(present_exps)}
exp_labels = {
    'shared_unperturbed': 'Unperturbed', 'dose_response_25pct': '25%',
    'dose_response_50pct': '50%', 'dose_response_75pct': '75%',
    'full_replacement': '100%', 'random_labels_weighted': 'Random label',
}
split = exp_pos.get('full_replacement', len(present_exps) - 1)
 
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey='row')
for ax, label in zip(axes.ravel(), wanted):
    ldf = pdf[pdf[L4].eq(label)].copy()
    if ldf.empty:
        ax.set_title(label)
        continue
    ldf['x'] = ldf['experiment'].map(exp_pos)
    ldf = ldf.sort_values('x')
 
    # dose-response segment (up to full_replacement)
    seg1 = ldf[ldf['x'] <= split]
    # random label segment (after full_replacement)
    seg2 = ldf[ldf['x'] > split]
 
    for seg in [seg1, seg2]:
        if seg.empty:
            continue
        ax.errorbar(seg['x'], seg['mean_lfc'], yerr=seg['sem_lfc'],
                    color='#4f278b', marker='o', linewidth=1.5,
                    elinewidth=1.0, capsize=3)
 
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, zorder=0)
    ax.set_title(label)
    ax.set_xticks(range(len(present_exps)))
    ax.set_xticklabels([exp_labels.get(e, e) for e in present_exps], rotation=90, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)
 
fig.supxlabel('Experiment')
fig.supylabel('log₂ FC of ICB DEG signature (vs unperturbed)')
fig.tight_layout()
fig.savefig(f"{output_dir}/log2FC_ICB_DEG_signature_dose_response.svg", bbox_inches="tight")
plt.close(fig)
 

In [24]:
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

line_lw = 0.25
line_color = "black"

def add_sig_bracket(ax, x1, x2, y, h, text, fontsize=10, lw=0.25):
    ax.plot(
        [x1, x1, x2, x2],
        [y,  y + h, y + h, y],
        color=line_color,
        lw=lw,
        clip_on=False
    )
    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color="black",
    )

ct = 'IFN+'
sig_genes = upreg_genes[ct]
# Recompute raw values using median for boxplot
box_raw_rows = []
for exp in ['shared_unperturbed', 'full_replacement', 'random_labels_weighted']:
    if exp not in mean_adatas:
        continue
    for fname, adata in mean_adatas[exp].items():
        section = Path(fname).stem.rsplit('__', 1)[0]
        if section not in SECTIONS:
            continue
        for label, val in mean_sig_by_celltype(adata, agg='median').items():
            box_raw_rows.append({'experiment': exp, 'section': section, L4: label, 'mean_raw': val})
box_raw_df = pd.DataFrame(box_raw_rows)
ifn_raw = box_raw_df[box_raw_df[L4] == ct]
ifn_pivot = ifn_raw.pivot(index='section', columns='experiment', values='mean_raw')
lfc_box_rows = []
for sec, row in ifn_pivot.iterrows():
    base = row.get('shared_unperturbed')
    if pd.isna(base):
        continue
    for exp, label in [('full_replacement', '100% perturbation'),
                       ('random_labels_weighted', 'Random relabel')]:
        if exp in row and pd.notna(row[exp]):
            lfc = np.log2((row[exp] + pseudo) / (base + pseudo))
            lfc_box_rows.append({'section': sec, 'condition': label, 'log2FC': lfc})
lfc_box_df = pd.DataFrame(lfc_box_rows)
paired = lfc_box_df.pivot(index='section', columns='condition', values='log2FC').dropna()
print(paired)
print()
for c in paired.columns:
    print(f'{c}: median log2FC = {paired[c].mean():.4f}')

fig, ax = plt.subplots(figsize=(4.5, 5.4), constrained_layout=True)
ax.grid(False)

conditions = ['Random relabel', '100% perturbation']
box_colors = {conditions[0]: '#4f278b', conditions[1]: '#fab673'}
data = [paired[c].to_numpy(dtype=float) for c in conditions]

bp = ax.boxplot(
    data,
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color=line_color, linewidth=line_lw),
    whiskerprops=dict(color=line_color, linewidth=line_lw),
    capprops=dict(color=line_color, linewidth=line_lw),
    boxprops=dict(edgecolor=line_color, linewidth=line_lw),
)
for box, cond in zip(bp["boxes"], conditions):
    box.set_facecolor(box_colors[cond])
    box.set_alpha(0.85)
    box.set_edgecolor(line_color)
    box.set_linewidth(line_lw)

for sec in paired.index:
    yvals = [paired.loc[sec, c] for c in conditions]
    ax.plot([1, 2], yvals, color=line_color, linewidth=line_lw, zorder=1)
    ax.scatter([1, 2], yvals, color="black", s=24, zorder=2)

ax.axhline(0, color=line_color, linestyle="--", linewidth=line_lw)
ax.set_xticks([1, 2])
ax.set_xticklabels(conditions, fontsize=10)
ax.set_title('IFN+ TAMs')
ax.set_ylabel('log₂ FC (vs unperturbed)')

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(line_lw)
    ax.spines[spine].set_color(line_color)
ax.tick_params(axis="both", colors="black", width=line_lw)

if len(paired) >= 4:
    stat, p = wilcoxon(paired[conditions[0]], paired[conditions[1]])
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    bracket_y = paired.values.max() + 0.02
    bracket_h = 0.01
    add_sig_bracket(ax, 1, 2, bracket_y, bracket_h,
                    f"{stars} (p={p:.3g})", fontsize=10, lw=line_lw)
else:
    ax.text(1.5, ax.get_ylim()[1] * 0.9, f'n={len(paired)} (too few)',
            ha='center', fontsize=8, color='gray')

fig.savefig(
    f"{output_dir}/lfc_boxplot_IFNplus_log2FC.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)
plt.close(fig)

condition          100% perturbation  Random relabel
section                                             
CV1-KID-0-FO-1              0.442800       -0.108641
CV1-KID-0-FT-2              0.238160        0.045515
CV5-KID-0-FO-1              0.394215       -0.031348
CV6-KID-0-FT-1              0.164834        0.024733
CV7-KID-0-FT-2-s3           0.346922       -0.009368
CV9-KID-0-FT-2              0.237536        0.002048
DI10-KID-0-FT-3             0.600365        0.233594
DI13-KID-0-FT-1             0.102124        0.085857

100% perturbation: median log2FC = 0.3159
Random relabel: median log2FC = 0.0303


In [25]:
# --- Dotplots: IFN+ TAMs signature genes across dose experiments ---

SECTIONS_ORIG = ['CV1-KID-0-FT-2', 'CV6-KID-0-FT-1', 'CV7-KID-0-FT-2-s3', 'CV9-KID-0-FT-2']
ct = 'IFN+'
gene_list = [
    "CD163", "MRC1", "MARCO", "LGALS9",
    "IFNGR1", "ITGAX", "LAMP1", "CXCR4",
    "CD276", "PVR",
    "AXL", "MERTK", "MAF", "LGMN",
]

output_dir_tam_dotplots = os.path.join(output_dir, "perturb_TAM_signature_dotplots")
os.makedirs(output_dir_tam_dotplots, exist_ok=True)

adatas_by_section = {}
available = None
for exp in EXP_ORDER:
    if exp not in mean_adatas:
        continue
    for fname, adata in sorted(mean_adatas[exp].items()):
        section = Path(fname).stem.rsplit('__', 1)[0]
        if section not in SECTIONS_ORIG:
            continue
        gen = adata.copy()
        gen.X = np.asarray(gen.obsm[GEN_KEY], dtype=np.float64)
        gen.obs[L4] = gen.obs[L4].astype(str).replace({'nan': 'NA', 'None': 'NA'}).values
        if available is None:
            available = [g for g in gene_list if g in gen.var_names]
        sub = gen[gen.obs[L4].eq(ct), available].copy()
        if sub.n_obs == 0:
            continue
        sub.X = np.clip(sub.X.astype(np.float64), 0, None)
        sub.obs['experiment'] = exp
        sub.obs_names = pd.Index([f"{exp}__{section}__{n}" for n in sub.obs_names.astype(str)])
        adatas_by_section.setdefault(section, []).append(sub)

for section in SECTIONS_ORIG:
    if section not in adatas_by_section:
        continue
    sec = ad.concat(adatas_by_section[section], join='outer', merge='same')
    sec.obs['experiment'] = pd.Categorical(
        sec.obs['experiment'].astype(str), categories=EXP_ORDER, ordered=True
    )
    sc.pp.log1p(sec)
    dp = sc.pl.dotplot(
        sec, var_names=available, groupby='experiment',
        categories_order=EXP_ORDER, use_raw=False, expression_cutoff=0.0,
        figsize=(max(len(available) * 0.6, 12), 4),
        title=f'{section} | {ct}', standard_scale='var',
        smallest_dot=0, return_fig=True,
    )
    dp.largest_dot = 300
    safe_section = section.replace('/', '_')
    dp.savefig(
        f"{output_dir_tam_dotplots}/{safe_section}_IFNplus_TAM_dotplot.svg",
        bbox_inches="tight",
    )
    plt.close('all')

## Spatial plot analysis

In [ ]:
# Load adata
adata_CV1_FT2 = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Perturbations/Control_workflow/mean_generated_adatas/shared_unperturbed/CV1-KID-0-FT-2__unperturbed.h5ad')
adata_CV1_FT2_perturbed = sc.read_h5ad('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Perturbations/Control_workflow/mean_generated_adatas/full_replacement/CV1-KID-0-FT-2__perturbed.h5ad')

In [ ]:
adata_CV1_FT2_perturbed

AnnData object with n_obs × n_vars = 16851 × 5001
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'batch', 'collection site type', 'donor', 'Xenium barcode', 'drug', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', '_scvi_batch', '_scvi_labels', 'SCVI_CLUSTERS_KEY', 'low_quality_cell_highlight', 'highlight', 'level_1_cell_type', 'level_2_cell_type', 'level_3_cell_type', 'cell_status', 'celltype_scanvi', 'C_scANVI', 'section', 'concate_key', 'level_4_cell_type'
    uns: 'run_mean_summary', 'spatial_neighbors'
    obsm: 'MintFLow_Generated_Xmic', 'spatial'
    layers: 'counts'
    obsp: 'spatial_connectivities', 'spatial_d

In [ ]:
adata_CV1_FT2.obs['level_3_cell_type'].value_counts()


level_3_cell_type
Tumour epithelium               4043
Tumour associated macrophage    3143
Tumour capillary vasculature    2782
CD8+ T lymphocyte               1719
CD4+ T lymphocyte               1464
Cancer associated fibroblast    1015
Activated endothelium            755
DN T lymphocyte                  707
Conventional dendritic cell      271
Vascular smooth muscle           245
Resident fibroblast              160
Pericyte                         155
Lymphatic vasculature            130
Plasma cell                      126
B lymphocyte                      75
Peritubular capillary             48
Kidney resident macrophage         8
Proximal tubule                    3
Juxtaglomerular cell               1
Distal tubule                      1
Name: count, dtype: int64

In [ ]:
adata_CV1_FT2_perturbed.obs['level_3_cell_type'].value_counts()

level_3_cell_type
Tumour epithelium               4043
Tumour associated macrophage    3143
Tumour capillary vasculature    2782
CD8+ T lymphocyte ICB           1719
CD4+ T lymphocyte ICB           1464
Cancer associated fibroblast    1015
Activated endothelium            755
DN T lymphocyte ICB              707
Conventional dendritic cell      271
Vascular smooth muscle           245
Resident fibroblast              160
Pericyte                         155
Lymphatic vasculature            130
Plasma cell                      126
B lymphocyte                      75
Peritubular capillary             48
Kidney resident macrophage         8
Proximal tubule                    3
Distal tubule                      1
Juxtaglomerular cell               1
Name: count, dtype: int64

In [ ]:
# generate new obs col 
map_dict = {
    'Tumour associated macrophage': 'TAM',
    'CD8+ T lymphocyte': 'T cell',
    'CD4+ T lymphocyte': 'T cell',
    'DN T lymphocyte': 'T cell',
    'CD8+ T lymphocyte ICB': 'ICB T cell',
    'CD4+ T lymphocyte ICB': 'ICB T cell',
    'DN T lymphocyte ICB': 'ICB T cell',
}

adata_CV1_FT2.obs['TCandTAM'] = adata_CV1_FT2.obs['level_3_cell_type'].copy()
adata_CV1_FT2_perturbed.obs['TCandTAM'] = adata_CV1_FT2_perturbed.obs['level_3_cell_type'].copy()

adata_CV1_FT2.obs['TCandTAM'] = adata_CV1_FT2.obs['TCandTAM'].map(map_dict)
adata_CV1_FT2_perturbed.obs['TCandTAM'] = adata_CV1_FT2_perturbed.obs['TCandTAM'].map(map_dict)

In [ ]:
assigned = {
    "T cell":   "#4f278b",  
    "ICB T cell":   "#fab673",
    "TAM":  "#74cf3c",
    "Other":    'lightgray',  # light gray
}

In [ ]:
output_dir_spatial = os.path.join(output_dir, "perturb_spatial_maps")
os.makedirs(output_dir_spatial, exist_ok=True)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2,
    color="TCandTAM",
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax,
    legend_loc=None,
    palette = assigned
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,                  
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)

plt.tight_layout()
fig.savefig(f"{output_dir_spatial}/unperturbed_cell_types.svg", bbox_inches="tight")
plt.close(fig)


fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2_perturbed,
    color="TCandTAM",
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax,
    legend_loc=None,
    palette = assigned
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,                  
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)


plt.tight_layout()
fig.savefig(f"{output_dir_spatial}/perturbed_cell_types.svg", bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_3263839/610410743.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3263839/610410743.py:35: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2,
    color="TCandTAM",
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax,
    legend_loc='right margin',
    palette = assigned
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,                  
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)

plt.tight_layout()
fig.savefig(f"{output_dir_spatial}/unperturbed_cell_types_legend.svg", bbox_inches="tight")
plt.close(fig)


fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2_perturbed,
    color="TCandTAM",
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax,
    legend_loc='right margin',
    palette = assigned
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,                  
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)


plt.tight_layout()
fig.savefig(f"{output_dir_spatial}/perturbed_cell_types_legend.svg", bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_3263839/297504198.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3263839/297504198.py:35: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


In [ ]:
def score_genes_from_obsm(adata, gene_list, score_name, obsm_key="MintFLow_Generated_Xmic"):
    """Score genes using an obsm matrix (log1p-transformed, no normalisation) instead of .X"""
    # Build a lightweight AnnData pointing at the obsm matrix, log1p-transformed
    expr = np.log1p(np.clip(np.asarray(adata.obsm[obsm_key], dtype=np.float64), 0, None))
    tmp = ad.AnnData(
        X=expr,
        obs=adata.obs,
        var=adata.var,
    )
    sc.tl.score_genes(tmp, gene_list=gene_list, score_name=score_name)
    # Write the score back
    adata.obs[score_name] = tmp.obs[score_name].values


In [ ]:
# IFN+ Macrophage signature
Signature = [
    'CD163', 'ANXA2', 'SEPTIN9', 'CYP1B1', 'PER1', 'LAMP1', 'IRS2', 'GRN', 'CHIT1',
    'RNF187', 'HOXB7', 'CYP27A1', 'CXCR4', 'SH3BP5', 'MRC1', 'MARCO', 'CALM3',
    'CCDC80', 'VEGFB', 'SOST', 'LGALS9', 'SMAP2', 'PALLD', 'IFNGR1', 'FSTL1',
    'AEBP1', 'F13A1', 'SPON2', 'PLA2G7', 'AOC3', 'CTSK', 'MMP9', 'ALDOA', 'ACE',
    'LAMB3', 'COL4A1', 'PLTP', 'ITGAX', 'CD99', 'LTBP2', 'REN', 'FRZB', 'TNXB',
    'LPL', 'COL5A1', 'SNAI2', 'G6PD', 'ZNF683', 'PGF', 'MAN1A1', 'UCHL1',
    'SERPINA3', 'SULF1', 'THBS2', 'LBP', 'C5AR1', 'ANXA5', 'THY1', 'CES1'
]

score_name = "IFN_signature_score"

# keep only genes present in each dataset
sig_unperturbed = [g for g in Signature if g in adata_CV1_FT2.var_names]
sig_perturbed   = [g for g in Signature if g in adata_CV1_FT2_perturbed.var_names]

# score genes
score_genes_from_obsm(adata_CV1_FT2, gene_list=sig_unperturbed, score_name=score_name)
score_genes_from_obsm(adata_CV1_FT2_perturbed, gene_list=sig_perturbed, score_name=score_name)

computing score 'IFN_signature_score'
    finished: added
    'IFN_signature_score', score of gene set (adata.obs).
    998 total control genes are used. (0:00:23)
computing score 'IFN_signature_score'
    finished: added
    'IFN_signature_score', score of gene set (adata.obs).
    1042 total control genes are used. (0:00:25)


In [ ]:

# shared color scale across both objects
combined_scores = np.concatenate([
    adata_CV1_FT2.obs[score_name].values,
    adata_CV1_FT2_perturbed.obs[score_name].values
])

# choose the percentile cap here
lower_pct = 5
upper_pct = 95

vmin = np.nanpercentile(combined_scores, lower_pct)
vmax = np.nanpercentile(combined_scores, upper_pct)

# unperturbed
fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2,
    color=score_name,
    cmap="plasma",
    vmin=vmin,
    vmax=vmax,
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)

plt.tight_layout()
fig.savefig(
    f"{output_dir_spatial}/unperturbed_IFN_signature_score_plasma_shared_scale_p{lower_pct}_p{upper_pct}.svg",
    bbox_inches="tight"
)
plt.close(fig)

# perturbed
fig, ax = plt.subplots(figsize=(3, 3))

sc.pl.spatial(
    adata_CV1_FT2_perturbed,
    color=score_name,
    cmap="plasma",
    vmin=vmin,
    vmax=vmax,
    size=1.5,
    spot_size=10,
    frameon=False,
    show=False,
    ax=ax
)

scalebar = AnchoredSizeBar(
    ax.transData,
    100,
    "100 µm",
    "lower right",
    pad=-0.7,
    color="black",
    frameon=False,
    size_vertical=2,
    fontproperties=fm.FontProperties(size=10),
)
ax.add_artist(scalebar)

plt.tight_layout()
fig.savefig(
    f"{output_dir_spatial}/perturbed_IFN_signature_score_plasma_shared_scale_p{lower_pct}_p{upper_pct}.svg",
    bbox_inches="tight"
)
plt.close(fig)

/tmp/ipykernel_3263839/3644923364.py:17: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3263839/3644923364.py:53: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(


# Macrophage depletion

## Generated adatas

In [ ]:
### The perturbation experiment is repeated 10 times with different random seeds

CONTROL_WORKFLOW_ROOT = Path(
    '/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Perturbations/Control_workflow_mac_deletion'
)
AGGREGATION_CACHE_DIR = CONTROL_WORKFLOW_ROOT / 'mean_generated_cache'
GENERATED_OBSM_KEY = 'MintFLow_Generated_Xmic'
GENERATED_XINT_OBSM_KEY = 'MintFlow_Generated_Xint'
TARGET_LEVEL4_LABEL_COLUMN = 'level_4_cell_type'
TARGET_LEVEL4_LABEL = 'LAG3+ IRF1+ IFN-related'
OBS_RUN_COUNT_COLUMN = 'n_runs_aggregated'
PREFERRED_EXPERIMENT_ORDER = [
    'shared_unperturbed',
    'dose_response_25pct',
    'dose_response_50pct',
    'dose_response_75pct',
    'full_deletion',
    'random_deletion',
    'spatial_shuffle_after_full_deletion',
]

def get_run_dirs(root=CONTROL_WORKFLOW_ROOT):
    return sorted(path for path in root.iterdir() if path.is_dir() and path.name.startswith('run_'))


def collect_generated_h5ad_map(run_dir, experiment_name):
    generated_dir = Path(run_dir) / experiment_name / 'generated_adatas'
    return {path.name: path for path in sorted(generated_dir.glob('*.h5ad'))}


def list_experiment_names(run_dirs, preferred_order=None):
    preferred_order = preferred_order or []
    experiment_names = sorted(
        set.intersection(
            *[
                {
                    path.name
                    for path in run_dir.iterdir()
                    if path.is_dir() and (path / 'generated_adatas').exists()
                }
                for run_dir in run_dirs
            ]
        )
    )
    ordered_names = [name for name in preferred_order if name in experiment_names]
    ordered_names.extend(name for name in experiment_names if name not in ordered_names)
    return ordered_names


def split_section_condition(filename):
    return Path(filename).stem.rsplit('__', 1)


def build_mean_generated_adata(
    source_paths,
    obsm_key=GENERATED_OBSM_KEY,
    label_column=TARGET_LEVEL4_LABEL_COLUMN,
    target_label=TARGET_LEVEL4_LABEL,
):
    expr_count = None
    expr_sums = {}
    obs_frame = None
    template_var = None
    template_var_names = None
    template_dtypes = {}
    obsm_keys_to_average = None

    for path in map(Path, source_paths):
        adata = sc.read_h5ad(path)

        if template_var is None:
            template_var = adata.var.copy()
            template_var_names = adata.var_names.copy()
            obsm_keys_to_average = [obsm_key]
            if GENERATED_XINT_OBSM_KEY in adata.obsm:
                obsm_keys_to_average.append(GENERATED_XINT_OBSM_KEY)
            template_dtypes = {key: np.asarray(adata.obsm[key]).dtype for key in obsm_keys_to_average}

        target_adata = adata[adata.obs[label_column].astype(str).eq(target_label)].copy()
        if target_adata.n_obs == 0:
            continue

        obs_names = target_adata.obs_names.astype(str)
        for key in obsm_keys_to_average:
            expr_df = pd.DataFrame(
                np.asarray(target_adata.obsm[key], dtype=np.float64),
                index=obs_names,
                columns=target_adata.var_names.astype(str),
            )
            expr_sums[key] = expr_df if key not in expr_sums else expr_sums[key].add(expr_df, fill_value=0.0)

        run_count = pd.Series(1.0, index=obs_names)
        expr_count = run_count if expr_count is None else expr_count.add(run_count, fill_value=0.0)

        current_obs = target_adata.obs.copy()
        current_obs.index = obs_names
        obs_frame = current_obs if obs_frame is None else obs_frame.combine_first(current_obs)

    if not expr_sums:
        mean_adata = sc.AnnData(
            X=np.zeros((0, template_var.shape[0]), dtype=np.float32),
            obs=pd.DataFrame(index=pd.Index([], dtype=str)),
            var=template_var.copy(),
        )
        for key in obsm_keys_to_average:
            mean_adata.obsm[key] = np.empty((0, template_var.shape[0]), dtype=template_dtypes[key])
        mean_adata.obs[OBS_RUN_COUNT_COLUMN] = pd.Series(dtype=int)
        return mean_adata

    valid_obs_names = expr_count.index
    mean_obs = obs_frame.reindex(valid_obs_names).copy()
    mean_obs[OBS_RUN_COUNT_COLUMN] = expr_count.reindex(valid_obs_names).astype(int).to_numpy()

    mean_adata = sc.AnnData(
        X=np.zeros((len(valid_obs_names), template_var.shape[0]), dtype=np.float32),
        obs=mean_obs,
        var=template_var.copy(),
    )
    for key in obsm_keys_to_average:
        mean_expr = expr_sums[key].reindex(valid_obs_names).div(expr_count, axis=0)
        mean_adata.obsm[key] = mean_expr.to_numpy(dtype=template_dtypes[key], copy=False)
    return mean_adata


def save_aggregated(mean_generated_adatas, cache_dir=AGGREGATION_CACHE_DIR):
    for experiment_name, file_dict in mean_generated_adatas.items():
        exp_dir = cache_dir / experiment_name
        exp_dir.mkdir(parents=True, exist_ok=True)
        for filename, adata in file_dict.items():
            adata.write_h5ad(exp_dir / filename)
    print(f'Saved aggregated adatas to {cache_dir}')


def load_aggregated(cache_dir=AGGREGATION_CACHE_DIR):
    mean_generated_adatas = {}
    for exp_dir in sorted(cache_dir.iterdir()):
        if not exp_dir.is_dir():
            continue
        mean_generated_adatas[exp_dir.name] = {
            path.name: sc.read_h5ad(path)
            for path in sorted(exp_dir.glob('*.h5ad'))
        }
    print(f'Loaded aggregated adatas from {cache_dir} ({len(mean_generated_adatas)} experiments)')
    return mean_generated_adatas


def get_aggregated_adata(aggregated_adatas, experiment_name, section_name, condition_name):
    filename = f'{section_name}__{condition_name}.h5ad'
    return aggregated_adatas[experiment_name][filename]


def list_sections(aggregated_adatas, experiment_name, condition_name):
    sections = []
    for filename in aggregated_adatas.get(experiment_name, {}):
        section_name, current_condition = split_section_condition(filename)
        if current_condition == condition_name:
            sections.append(section_name)
    return sorted(sections)


In [27]:
if AGGREGATION_CACHE_DIR.exists() and any(AGGREGATION_CACHE_DIR.iterdir()):
    mean_generated_adatas = load_aggregated()
else:
    run_dirs = get_run_dirs()
    run_names = [run_dir.name for run_dir in run_dirs]
    experiment_names = list_experiment_names(run_dirs, preferred_order=PREFERRED_EXPERIMENT_ORDER)

    mean_generated_adatas = {}

    for experiment_name in experiment_names:
        run_file_maps = {
            run_name: collect_generated_h5ad_map(run_dir, experiment_name)
            for run_name, run_dir in zip(run_names, run_dirs)
        }
        all_filenames = sorted(set.intersection(*(set(file_map) for file_map in run_file_maps.values())))
        mean_generated_adatas[experiment_name] = {}

        for filename in tqdm(all_filenames, desc=experiment_name, leave=False):
            mean_adata = build_mean_generated_adata([run_file_maps[run_name][filename] for run_name in run_names])
            mean_generated_adatas[experiment_name][filename] = mean_adata

    save_aggregated(mean_generated_adatas)


Loaded aggregated adatas from /lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Perturbations/Control_workflow_mac_deletion/mean_generated_cache (7 experiments)


## Survival analysis 

In [28]:
SURVIVAL_EXPR_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/HiSeqV2')
SURVIVAL_CLIN_PATH = Path('/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Survival_analysis/survival-KIRC_survival.txt')
TARGET_SURVIVAL_CELL_TYPE = TARGET_LEVEL4_LABEL
CELL_SELECTION_LABEL = 'all aggregated cells (no MCC mask)'
MIN_DE_CELLS_PER_GROUP = 3
N_DE_GENES_TO_PRINT = 10000
DE_PVALUE_THRESHOLD = 0.05
DE_PVALUE_COLUMN = 'pvals_adj'
DISPLAY_DE_GENES = 10


def load_survival_inputs():
    expr_data = pd.read_csv(SURVIVAL_EXPR_PATH, sep='\t', index_col=0)
    clin_data = pd.read_csv(SURVIVAL_CLIN_PATH, sep='\t', index_col=0)

    expr_data = expr_data.loc[~expr_data.index.duplicated(keep='first')]
    common_samples = expr_data.columns.intersection(clin_data.index)
    expr_data = expr_data.loc[:, common_samples]
    clin_data = clin_data.loc[common_samples].copy()

    clin_data['OS.time'] = pd.to_numeric(clin_data['OS.time'], errors='coerce')
    clin_data['OS'] = pd.to_numeric(clin_data['OS'], errors='coerce')
    return expr_data, clin_data


def compute_cox_hr(expr_df, clin_df, genes):
    """Compute Cox PH hazard ratio and 95% CI for a gene signature."""
    genes_present = [gene for gene in genes if gene in expr_df.index]
    if len(genes_present) < 2:
        return {'n_genes': len(genes_present), 'n': 0, 'HR': np.nan,
                'HR_lower': np.nan, 'HR_upper': np.nan, 'cox_p': np.nan}

    module_score = expr_df.loc[genes_present].mean(axis=0)
    df = clin_df.copy()
    df['module_score'] = module_score.reindex(df.index)
    df = df.dropna(subset=['OS.time', 'OS', 'module_score'])

    cph_df = df[['OS.time', 'OS', 'module_score']].rename(columns={'OS.time': 'T', 'OS': 'E'})
    try:
        cph = CoxPHFitter()
        cph.fit(cph_df, duration_col='T', event_col='E')
        hr = float(np.exp(cph.params_['module_score']))
        ci = np.exp(cph.confidence_intervals_)
        hr_lower = float(ci.loc['module_score'].iloc[0])
        hr_upper = float(ci.loc['module_score'].iloc[1])
        cox_p = float(cph.summary.loc['module_score', 'p'])
    except Exception:
        return {'n_genes': len(genes_present), 'n': len(df), 'HR': np.nan,
                'HR_lower': np.nan, 'HR_upper': np.nan, 'cox_p': np.nan}

    return {'n_genes': len(genes_present), 'n': len(df), 'HR': hr,
            'HR_lower': hr_lower, 'HR_upper': hr_upper, 'cox_p': cox_p}


def plot_forest(survival_df, experiment_title, xlim=None):
    """Forest plot of hazard ratios with 95% CI, two rows per section."""
    from matplotlib.lines import Line2D
    from matplotlib.ticker import FixedLocator, FixedFormatter, LogLocator, NullFormatter

    df = survival_df.dropna(subset=['HR']).reset_index(drop=True)
    if df.empty:
        print(f'No valid Cox results for {experiment_title}')
        return

    n_rows = len(df)
    fig, ax = plt.subplots(figsize=(10, max(3, n_rows * 0.45 + 1.5)))

    colors = {'unperturbed': '#1f77b4', 'perturbed': '#d62728'}
    y_positions = list(range(n_rows - 1, -1, -1))

    for y, (_, row) in zip(y_positions, df.iterrows()):
        color = colors.get(row['condition'], 'gray')
        ax.plot([row['HR_lower'], row['HR_upper']], [y, y],
                color=color, linewidth=2, solid_capstyle='round')
        ax.plot(row['HR'], y, 'o', color=color, markersize=7, zorder=5)

    ax.axvline(x=1.0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(FixedLocator([0.5, 1.0, 2.0]))
    ax.xaxis.set_major_formatter(FixedFormatter(['0.5', '1.0', '2.0']))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs='auto', numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    if xlim is not None:
        ax.set_xlim(xlim)

    labels = [f"{row['section']} \u2014 {row['condition']}" for _, row in df.iterrows()]
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=7)
    ax.tick_params(axis='x', labelsize=7)
    ax.set_xlabel('Hazard Ratio (95% CI, log scale)', fontsize=7)
    ax.set_title(
        f'{experiment_title} | {TARGET_SURVIVAL_CELL_TYPE}\n'
        f'Cox PH: pos-logFC genes ({DE_PVALUE_COLUMN} < {DE_PVALUE_THRESHOLD}) | {CELL_SELECTION_LABEL}',
        fontsize=7,
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)

    for y, (_, row) in zip(y_positions, df.iterrows()):
        sig = ' *' if row['cox_p'] < 0.05 else ''
        text = (f"HR={row['HR']:.2f} [{row['HR_lower']:.2f}\u2013{row['HR_upper']:.2f}] "
                f"p={row['cox_p']:.2e}{sig}  ({row['n_genes']}g)")
        ax.annotate(text, xy=(1.02, y), xycoords=('axes fraction', 'data'),
                    fontsize=7, va='center', ha='left', annotation_clip=False)

    legend_elements = [
        Line2D([0], [0], color='#1f77b4', marker='o', linestyle='-', label=' '),
        Line2D([0], [0], color='#d62728', marker='o', linestyle='-', label=' '),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=7, handletextpad=0, labelspacing=0.3)

    fig.tight_layout()
    fig.subplots_adjust(right=0.6)
    safe_title = experiment_title.replace('/', '_').replace(' ', '_')
    fig.savefig(
        f"{output_dir}/forest_{safe_title}.svg",
        bbox_inches='tight',
    )
    plt.close(fig)


def get_significant_positive_lfc_genes(df, p_thresh=DE_PVALUE_THRESHOLD, pval_col=DE_PVALUE_COLUMN):
    mask = (df['logfoldchanges'] > 0) & (df[pval_col] < p_thresh)
    return df.loc[mask, 'names'].dropna().drop_duplicates().astype(str).tolist()


def run_de_survival_analysis(experiment_name, perturbation_label, expr_data, clin_data):
    """Run DE + Cox survival analysis for one perturbation experiment vs shared_unperturbed."""
    sections = sorted(
        set(list_sections(mean_generated_adatas, 'shared_unperturbed', 'unperturbed'))
        & set(list_sections(mean_generated_adatas, experiment_name, 'perturbed'))
    )

    deg_results = {'perturbed': {}, 'unperturbed': {}}
    survival_rows = []

    for section_name in sections:
        adata_orig = get_aggregated_adata(mean_generated_adatas, 'shared_unperturbed', section_name, 'unperturbed')
        adata_pert = get_aggregated_adata(mean_generated_adatas, experiment_name, section_name, 'perturbed')

        n_orig, n_pert = adata_orig.n_obs, adata_pert.n_obs
        print(f'\n{section_name}: {CELL_SELECTION_LABEL} -> original={n_orig}, {experiment_name}={n_pert}')

        if min(n_orig, n_pert) < MIN_DE_CELLS_PER_GROUP:
            print(f'Skipping {section_name}: fewer than {MIN_DE_CELLS_PER_GROUP} cells in one group')
            continue

        adata_de = sc.AnnData(
            X=np.concatenate([
                np.asarray(adata_orig.obsm[GENERATED_OBSM_KEY], dtype=np.float64),
                np.asarray(adata_pert.obsm[GENERATED_OBSM_KEY], dtype=np.float64),
            ], axis=0),
            obs=pd.DataFrame({'original_vs_perturbed': ['original'] * n_orig + ['perturbed'] * n_pert}),
            var=adata_orig.var.copy(),
        )

        adata_de.layers['xspl_before_log1p'] = adata_de.X.copy()
        sc.pp.log1p(adata_de)

        sc.tl.rank_genes_groups(adata_de, groupby='original_vs_perturbed', groups=['perturbed'],
                                reference='original', method='wilcoxon', n_genes=adata_de.n_vars)
        df_pert = sc.get.rank_genes_groups_df(adata_de, group='perturbed')

        sc.tl.rank_genes_groups(adata_de, groupby='original_vs_perturbed', groups=['original'],
                                reference='perturbed', method='wilcoxon', n_genes=adata_de.n_vars,
                                key_added='rank_genes_groups_unperturbed')
        df_unpert = sc.get.rank_genes_groups_df(adata_de, group='original', key='rank_genes_groups_unperturbed')

        deg_results['perturbed'][section_name] = df_pert
        deg_results['unperturbed'][section_name] = df_unpert

        sig_unpert = get_significant_positive_lfc_genes(df_unpert)
        sig_pert = get_significant_positive_lfc_genes(df_pert)

        result_unpert = compute_cox_hr(expr_data, clin_data, sig_unpert)
        result_pert = compute_cox_hr(expr_data, clin_data, sig_pert)

        survival_rows.append({'section': section_name, 'condition': 'unperturbed', **result_unpert})
        survival_rows.append({'section': section_name, 'condition': 'perturbed', **result_pert})
        print(pd.DataFrame(survival_rows[-2:]).to_string(index=False))

        del adata_de
        gc.collect()

    survival_df = pd.DataFrame(survival_rows)
    return deg_results, survival_df

In [29]:
expr_data, clin_data = load_survival_inputs()

analyses = [
    ('full_deletion', 'Full-deletion'),
    #('random_deletion', 'Random-deletion'),
]

all_deg_results = {}
all_survival_dfs = {}

for experiment_name, label in analyses:
    display(Markdown(f'## {label}'))
    deg_results, survival_df = run_de_survival_analysis(experiment_name, label, expr_data, clin_data)
    all_deg_results[experiment_name] = deg_results
    all_survival_dfs[experiment_name] = survival_df
    display(survival_df)

# Compute shared x-axis limits across both forest plots
all_hr_values = pd.concat(all_survival_dfs.values(), ignore_index=True).dropna(subset=['HR'])
shared_xlim = (
    all_hr_values[['HR_lower', 'HR']].min().min() * 0.8,
    all_hr_values[['HR_upper', 'HR']].max().max() * 1.2,
)

# Replot both forest plots with the same scale
for experiment_name, label in analyses:
    plot_forest(all_survival_dfs[experiment_name], label, xlim=shared_xlim)

## Full-deletion


CV1-KID-0-FT-2: all aggregated cells (no MCC mask) -> original=779, full_deletion=779


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:04)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:01)
       section   condition  n_genes  n  HR  HR_lower  HR_upper  cox_p
CV1-KID-0-FT-2 unperturbed        0  0 NaN       NaN       NaN    NaN
CV1-KID-0-FT-2   perturbed        0  0 NaN       NaN       NaN    NaN

CV6-KID-0

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:09)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:09)
          section   condition  n_genes   n       HR  HR_lower  HR_upper    cox_p
CV7-KID-0-FT-2-s3 unperturbed      270 606 1.334634   1.03239  1.725364 0.027576
CV7-KID-0-FT-2-s3   perturbed      313 606 1.189771   0.83

/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:15)
ranking genes
    finished: added to `.uns['rank_genes_groups_unperturbed']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:21)
       section   condition  n_genes   n       HR  HR_lower  HR_upper    cox_p
CV9-KID-0-FT-2 unperturbed      782 606 1.721014  1.225687  2.416515 0.001718
CV9-KID-0-FT-2   perturbed     1136 606 0.796738  0.483207  1.31

,section,condition,n_genes,n,HR,HR_lower,HR_upper,cox_p
0,CV1-KID-0-FT-2,unperturbed,0,0,NaN,NaN,NaN,NaN
1,CV1-KID-0-FT-2,perturbed,0,0,NaN,NaN,NaN,NaN
2,CV6-KID-0-FT-1,unperturbed,0,0,NaN,NaN,NaN,NaN
3,CV6-KID-0-FT-1,perturbed,0,0,NaN,NaN,NaN,NaN
4,CV7-KID-0-FT-2-s3,unperturbed,270,606,1.334634,1.032390,1.725364,0.027576
5,CV7-KID-0-FT-2-s3,perturbed,313,606,1.189771,0.835200,1.694869,0.335813
6,CV9-KID-0-FT-2,unperturbed,782,606,1.721014,1.225687,2.416515,0.001718
7,CV9-KID-0-FT-2,perturbed,1136,606,0.796738,0.483207,1.313707,0.373157


In [30]:
dotplot_experiments = [
    "shared_unperturbed",
    "dose_response_25pct",
    "dose_response_50pct",
    "dose_response_75pct",
    "full_deletion",
    "random_deletion"
]

dotplot_condition_by_experiment = {
    "shared_unperturbed": "unperturbed",
    "dose_response_25pct": "perturbed",
    "dose_response_50pct": "perturbed",
    "dose_response_75pct": "perturbed",
    "full_deletion": "perturbed",
    "random_deletion": "perturbed",
}

dotplot_genes = [
    "CTLA4", "TIGIT", "BTLA", "CD274", "PDCD1LG2", "VSIR", "VSIG4", "LGALS9"
]

section_sets = [
    set(list_sections(mean_generated_adatas, exp, dotplot_condition_by_experiment[exp]))
    for exp in dotplot_experiments
]
dotplot_sections = sorted(set.intersection(*section_sets))

dotplot_adata_cache = {}
for section_name in dotplot_sections:
    for experiment_name in dotplot_experiments:
        condition_name = dotplot_condition_by_experiment[experiment_name]
        dotplot_adata_cache[(section_name, experiment_name)] = get_aggregated_adata(
            mean_generated_adatas, experiment_name, section_name, condition_name,
        )

eligible_sections = [
    s for s in dotplot_sections
    if all(dotplot_adata_cache[(s, exp)].n_obs > 0 for exp in dotplot_experiments)
]

reference_var = dotplot_adata_cache[(eligible_sections[0], dotplot_experiments[0])].var.copy()
reference_var["requested_gene"] = reference_var.index.astype(str)
available_genes = set(reference_var.index.astype(str))
resolved_dotplot_genes = [g for g in dotplot_genes if g in available_genes]

missing_genes = [g for g in dotplot_genes if g not in available_genes]
if missing_genes:
    print("Omitting missing genes:", ", ".join(missing_genes))

output_dir_mac_deletion_dotplots = os.path.join(output_dir, "mac_deletion_dotplots")
os.makedirs(output_dir_mac_deletion_dotplots, exist_ok=True)

for section_name in eligible_sections:
    raw_blocks, obs_blocks = [], []

    for experiment_name in dotplot_experiments:
        adata = dotplot_adata_cache[(section_name, experiment_name)]
        raw_expr = np.clip(np.asarray(adata.obsm[GENERATED_OBSM_KEY], dtype=np.float64), 0, None)
        obs_index = pd.Index([f"{experiment_name}::{n}" for n in adata.obs_names.astype(str)], dtype=str)
        raw_blocks.append(raw_expr)
        obs_blocks.append(pd.DataFrame({"experiment": experiment_name}, index=obs_index))

    section_raw = np.concatenate(raw_blocks, axis=0)
    section_obs = pd.concat(obs_blocks, axis=0)
    section_obs["experiment"] = pd.Categorical(section_obs["experiment"], categories=dotplot_experiments, ordered=True)

    section_adata = sc.AnnData(
        X=np.log1p(section_raw),
        obs=section_obs,
        var=reference_var.copy(),
    )
    section_adata.layers["generated_raw_clipped"] = section_raw

    dp = sc.pl.dotplot(
        section_adata,
        var_names=resolved_dotplot_genes,
        gene_symbols="requested_gene",
        groupby="experiment",
        layer="generated_raw_clipped",
        standard_scale="var",
        mean_only_expressed=True,
        figsize=(12, 4),
        title=f"{section_name} | {TARGET_LEVEL4_LABEL}",
        smallest_dot=0,
        return_fig=True,
    )
    dp.largest_dot = 500
    safe_section = section_name.replace('/', '_')
    dp.savefig(
        f"{output_dir_mac_deletion_dotplots}/{safe_section}_LAG3_IRF1_dotplot.svg",
        bbox_inches="tight",
    )
    plt.close('all')

    del section_adata
    gc.collect()